# V027 draft: training-only neighbor feasibility

**DRAFT — not frozen, not executed.** This notebook defines one bounded CPU diagnostic using only approved Gym training data. It does not train or infer a neural model. No original A-validation features or labels enter sampling, pools, statistics, retrieval or scores; its IDs are used only to enforce exclusion.

Sample 64 queries per R/C inside original A-fit, disjoint from every neighbor/reference/statistics pool. A fixed top-five input-neighbor pressure median is compared with a pool-only R/C × position inspiratory median. Use the complete known 80-step inputs, which is valid for this offline task. IDs identify audit artifacts only and are not retrieval inputs. Query pressure labels are reserved for diagnostic scoring.

Neighbor pressure MAD/IQR measures disagreement between retrieved training curves, **not calibrated predictive uncertainty**. Descriptive distance/error correlations and bins do not select new parameters. A positive diagnostic cannot establish benefit over the LSTM or trigger GPU training by itself; it supports considering a separately frozen cross-fitted residual-context experiment. No numerical goal or official score is promised.

**Weak-baseline limitation:** outperforming an R/C-position median can be much easier than outperforming the LSTM. Only native inspiratory mean MAE is the primary comparison; median error, disagreement, bins and correlations are descriptive. A synthetic query-label mutation invariance check is defined and required before any approved-data diagnostic execution. Per-R/C phase-mismatch fractions will expose retrievals with differing valve phases.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv, gc, hashlib, json, platform, time
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits, threadpool_info

V027_RECIPE = {'query_seed': 20262701, 'queries_per_RC': 64, 'expected_queries': 576, 'expected_Afit_breaths': 61114, 'expected_original_Aval_breaths': 6791, 'breath_steps': 80, 'neighbor_count': 5, 'query_batch_size': 32, 'retrieval_inputs': ['u_in', 'time_step', 'u_out'], 'distance': 'Euclidean RMS of all240 standardized entries of the complete80-step three-channel trace; equal channel/step weights; sameR/C pool only', 'standardization': 'One global mean and population std per input channel, fitted on pool rows only; zero std replaced by1; no target or query statistics', 'phase_handling': 'Known binary u_out enters distance as a standardized input channel; no exact-mask restriction or query exclusion. Score query u_out==0 only; report retrieved-neighbor phase mismatch.', 'prediction': 'Coordinatewise median of the five retrieved pool pressure curves', 'baseline': 'Pool-only median pressure for eachR/C and position using pool u_out==0 rows; empty cells fall back to thatR/C pool inspiratory median', 'target_dtype': 'float32 parsed only for originalAfit breaths; errors and aggregate scores float64', 'input_dtype': 'float64, parsed only for originalAfit breaths', 'ties': 'Stable sort by squared distance; exact ties retain canonical pool row order. Neither IDs nor pressures are passed to retrieval.', 'bootstrap_seed': 20262702, 'bootstrap_draws': 1000, 'bootstrap_unit': 'Whole query breath, point-weighted native inspiratory MAE difference, KNN minus baseline', 'distance_bins': 4, 'uncertainty_bins': 4, 'threads': 1}

def v027_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def v027_code(path):
    obj=json.loads(Path(path).read_text())
    return '\n'.join(''.join(c['source']) for c in obj['cells'] if c['cell_type']=='code')

def v027_require(ok,message):
    if not bool(ok):raise ValueError(message)

def v027_json(path,obj):
    path=Path(path)
    if path.exists():raise FileExistsError('Preserve existing artifact: '+str(path))
    path.write_text(json.dumps(obj,indent=2,allow_nan=False)+'\n')

def v027_array(path,value):
    path=Path(path)
    if path.exists():raise FileExistsError('Preserve existing artifact: '+str(path))
    with path.open('xb') as f:np.save(f,value,allow_pickle=False)

def v027_ah(value):
    a=np.ascontiguousarray(value)
    h=hashlib.sha256(str((a.shape,a.dtype.str)).encode())
    h.update(memoryview(a).cast('B'))
    return h.hexdigest()


## Fit-only loading and deterministic inner holdout


In [2]:
def v027_load_Afit_only(train_csv,Afit_ids):
    """Filter by breath ID before parsing any input or pressure value.

    Original A-validation and every other non-fit row are skipped after reading
    only their ID for membership. Their raw CSV text is streamed but their
    features/labels never enter arrays, statistics, sampling or scores.
    """
    bids=np.sort(np.asarray(Afit_ids,dtype=np.int64))
    v027_require(len(np.unique(bids))==61114,'Afit membership changed')
    at={int(b):i for i,b in enumerate(bids)}
    x=np.empty((len(bids),80,3),dtype=np.float64)
    y=np.empty((len(bids),80),dtype=np.float32)
    rows=np.empty((len(bids),80),dtype=np.int64)
    rc=np.empty((len(bids),2),dtype=np.int16)
    count=np.zeros(len(bids),dtype=np.int16)
    parsed_fit_rows=0
    with Path(train_csv).open(newline='') as f:
        reader=csv.DictReader(f)
        v027_require(set(['id','breath_id','R','C','u_in','time_step','u_out','pressure']).issubset(reader.fieldnames),'Approved schema changed')
        for row in reader:
            i=at.get(int(row['breath_id']))
            if i is None:continue
            j=int(count[i]);v027_require(j<80,'More than80 rows in an Afit breath')
            setting=(int(row['R']),int(row['C']))
            if j==0:rc[i]=setting
            else:v027_require(tuple(rc[i])==setting,'R/C changes inside an Afit breath')
            rows[i,j]=int(row['id'])
            x[i,j]=[float(row['u_in']),float(row['time_step']),float(row['u_out'])]
            y[i,j]=np.float32(float(row['pressure']))
            count[i]+=1;parsed_fit_rows+=1
    v027_require(np.all(count==80) and parsed_fit_rows==61114*80,'Afit row coverage changed')
    v027_require(np.isfinite(x).all() and np.isfinite(y).all(),'Nonfinite Afit input/pressure')
    v027_require(np.isin(x[:,:,2],[0.,1.]).all(),'Nonbinary known phase')
    v027_require((np.diff(x[:,:,1],axis=1)>=0).all(),'Nonmonotone Afit time')
    v027_require((np.diff(rows,axis=1)>0).all(),'Afit row order changed')
    v027_require(set(map(tuple,rc))=={(r,c) for r in [5,20,50] for c in [10,20,50]},'UnexpectedR/C values')
    return bids,x,y,rows,rc

def v027_query_pool_indices(rc,seed,per_RC):
    rng=np.random.default_rng(seed);selected=[]
    for setting in sorted(set(map(tuple,rc))):
        group=np.flatnonzero(np.all(rc==setting,axis=1))
        v027_require(len(group)>=per_RC+5,'Insufficient sameR/C pool/query population')
        selected.extend(rng.choice(group,size=per_RC,replace=False).tolist())
    query=np.sort(np.asarray(selected,dtype=np.int64))
    pool=np.setdiff1d(np.arange(len(rc)),query,assume_unique=True)
    v027_require(len(query)==576 and len(pool)==60538 and not np.intersect1d(query,pool).size,'Query/pool partition changed')
    return query,pool


## Input-only retrieval and pool-only reference


In [3]:
def v027_retrieve(input_query,input_pool,RC_query,RC_pool,k=5,batch_size=32):
    """Retrieval sees known-input arrays and lung settings only; no targets/IDs."""
    v027_require(input_query.ndim==input_pool.ndim==3 and input_query.shape[1:]==input_pool.shape[1:]==(80,3),'Input shape changed')
    mu=input_pool.mean(axis=(0,1),dtype=np.float64)
    sd=input_pool.std(axis=(0,1),dtype=np.float64)
    sd=np.where(sd>0,sd,1.)
    pool_features=((input_pool-mu)/sd).reshape(len(input_pool),240)
    query_features=((input_query-mu)/sd).reshape(len(input_query),240)
    nearest=np.empty((len(input_query),k),dtype=np.int64)
    distance=np.empty((len(input_query),k),dtype=np.float64)
    for setting in sorted(set(map(tuple,RC_query))):
        qi=np.flatnonzero(np.all(RC_query==setting,axis=1))
        pi=np.flatnonzero(np.all(RC_pool==setting,axis=1))
        v027_require(len(pi)>=k,'Insufficient sameR/C neighbors')
        pp=pool_features[pi];pn=np.einsum('ij,ij->i',pp,pp)
        for start in range(0,len(qi),batch_size):
            use=qi[start:start+batch_size];qq=query_features[use]
            d2=np.einsum('ij,ij->i',qq,qq)[:,None]+pn[None,:]-2.*(qq@pp.T)
            d2=np.maximum(d2,0.)/240.
            # Stable ordering is deterministic for exact equal distances.
            # Pool row order is canonical; IDs and pressures are not consulted.
            order=np.argsort(d2,axis=1,kind='stable')[:,:k]
            nearest[use]=pi[order]
            distance[use]=np.sqrt(np.take_along_axis(d2,order,axis=1))
    return nearest,distance,dict(mean=mu.tolist(),population_std=sd.tolist(),source='pool known inputs only')

def v027_baseline(pool_pressure,pool_phase,RC_pool,RC_query):
    """Query pressures are not accepted by this training-pool-only estimator."""
    curves={};fallback={}
    for setting in sorted(set(map(tuple,RC_query))):
        group=np.all(RC_pool==setting,axis=1)
        yy=pool_pressure[group];mask=pool_phase[group]==0
        prior=float(np.median(yy[mask].astype(np.float64)))
        curve=[];missing=[]
        for step in range(80):
            values=yy[:,step][mask[:,step]].astype(np.float64)
            curve.append(float(np.median(values)) if len(values) else prior)
            if not len(values):missing.append(step)
        curves[setting]=np.asarray(curve,dtype=np.float64);fallback[str(setting)]=missing
    return np.stack([curves[tuple(v)] for v in RC_query]),fallback

def v027_spearman(a,b):
    a=np.asarray(a,dtype=np.float64);b=np.asarray(b,dtype=np.float64)
    v027_require(a.shape==b.shape and np.isfinite(a).all() and np.isfinite(b).all(),'Invalid correlation inputs')
    if len(a)<3 or np.ptp(a)==0 or np.ptp(b)==0:return None
    ra=pd.Series(a).rank(method='average').to_numpy();rb=pd.Series(b).rank(method='average').to_numpy()
    return float(np.corrcoef(ra,rb)[0,1])

def v027_error_table(error,phase,rc,prediction_name):
    records=[];mask=phase==0;all_total=float(error[mask].sum())
    for setting in sorted(set(map(tuple,rc))):
        select=np.all(rc==setting,axis=1);e=error[select][mask[select]]
        records.append(dict(model=prediction_name,R=int(setting[0]),C=int(setting[1]),breaths=int(select.sum()),points=len(e),MAE=float(e.mean()),total_absolute_error=float(e.sum()),fraction_model_total_error=float(e.sum()/all_total)))
    return records

def v027_bins(values,errors,point_counts,bins):
    """Descriptive quantile bins only; they never choose weights or queries."""
    values=np.asarray(values);errors=np.asarray(errors);point_counts=np.asarray(point_counts)
    edges=np.unique(np.quantile(values,np.linspace(0,1,bins+1)))
    if len(edges)<2:return [dict(bin=0,count=len(values),points=int(point_counts.sum()),MAE=float(errors.sum()/point_counts.sum()),value_min=float(values.min()),value_max=float(values.max()))]
    which=np.searchsorted(edges[1:-1],values,side='right');records=[]
    for group in range(len(edges)-1):
        take=which==group
        if not take.any():continue
        records.append(dict(bin=group,count=int(take.sum()),points=int(point_counts[take].sum()),MAE=float(errors[take].sum()/point_counts[take].sum()),value_min=float(values[take].min()),value_max=float(values[take].max())))
    return records

def v027_predict_from_pool(input_query,input_pool,RC_query,RC_pool,pool_pressure):
    """Prediction API deliberately has no query-label parameter."""
    nn,distance,scaler=v027_retrieve(input_query,input_pool,RC_query,RC_pool,k=5,batch_size=32)
    neighbor_pressure=pool_pressure[nn].astype(np.float64)
    prediction=np.median(neighbor_pressure,axis=1)
    disagreement=np.median(np.abs(neighbor_pressure-prediction[:,None,:]),axis=1)
    spread=np.quantile(neighbor_pressure,.75,axis=1)-np.quantile(neighbor_pressure,.25,axis=1)
    baseline,fallback=v027_baseline(pool_pressure,input_pool[:,:,2],RC_pool,RC_query)
    return dict(nn=nn,distance=distance,scaler=scaler,prediction=prediction,disagreement=disagreement,spread=spread,baseline=baseline,fallback=fallback)

def v027_synthetic_preflight():
    """Synthetic-only leakage/sensitivity checks; not an approved-data score."""
    rng=np.random.default_rng(270027);steps=np.arange(80,dtype=np.float64)
    pool=np.zeros((7,80,3),dtype=np.float64)
    pool[:,:,0]=rng.normal(size=(7,80))
    pool[:,:,1]=steps[None,:]*.03
    pool[:,:,2]=(steps[None,:]>=np.arange(30,37)[:,None]).astype(float)
    query=pool[[0,3]].copy();query[:,:,0]+=.03
    rc_pool=np.tile([20,20],(7,1));rc_query=np.tile([20,20],(2,1))
    labels=(np.arange(7)[:,None]+steps[None,:]*.1).astype(np.float32)
    fake_query_labels=np.zeros((2,80),dtype=np.float32)
    first=v027_predict_from_pool(query,pool,rc_query,rc_pool,labels)
    first_score=float(np.abs(first['prediction']-fake_query_labels).mean())
    fake_query_labels[:]=1000.
    second=v027_predict_from_pool(query,pool,rc_query,rc_pool,labels)
    changed_pool=v027_predict_from_pool(query,pool,rc_query,rc_pool,labels+10.)
    checks={
        'fake_query_label_mutation_leaves_neighbor_selection_unchanged':np.array_equal(first['nn'],second['nn']),
        'fake_query_label_mutation_leaves_distances_and_scaler_unchanged':np.array_equal(first['distance'],second['distance']) and first['scaler']==second['scaler'],
        'fake_query_label_mutation_leaves_both_predictions_unchanged':np.array_equal(first['prediction'],second['prediction']) and np.array_equal(first['baseline'],second['baseline']),
        'fake_query_label_mutation_changes_only_scoring':first_score!=float(np.abs(second['prediction']-fake_query_labels).mean()),
        'pool_pressure_mutation_does_not_change_input_retrieval':np.array_equal(first['nn'],changed_pool['nn']) and np.array_equal(first['distance'],changed_pool['distance']),
        'pool_pressure_mutation_changes_predictions':np.allclose(changed_pool['prediction'],first['prediction']+10.,atol=1e-6,rtol=0),
        'exact_top5_neighbors_and_finite_predictions':first['nn'].shape==(2,5) and np.isfinite(first['prediction']).all() and np.isfinite(first['baseline']).all(),
    }
    for name,ok in checks.items():v027_require(ok,'Synthetic preflight failed: '+name)
    return dict(status='passed',check_count=len(checks),checks={k:bool(v) for k,v in checks.items()},synthetic_only=True,approved_data_reads=0,gradient_model_fits=0,neural_checkpoint_inferences=0)


## Explicit frozen-protocol runner


In [4]:
def run_neighbors_feasibility_v027(root='.',*,frozen_protocol_path='research/neighbors_protocol_v027.json',expected_protocol_sha256,expected_source_code_sha256):
    """Inert by default. Root must review/freeze a protocol before calling.

    A positive diagnostic does NOT authorize/trigger GPU fitting automatically.
    The returned score is a training-only inner holdout result, not original-A,
    cold-F, official MAE, or evidence of improvement over the LSTM.
    """
    root=Path(root).resolve();protocol=root/frozen_protocol_path
    v027_require(v027_sha(protocol)==expected_protocol_sha256,'Frozen protocol changed')
    plan=json.loads(protocol.read_text())
    v027_require(plan['status']=='frozen' and plan['recipe']==V027_RECIPE,'Draft or unreviewed recipe cannot run')
    source=root/'research/neighbors_feasibility_v027.ipynb'
    source_hash=hashlib.sha256(v027_code(source).encode()).hexdigest()
    v027_require(source_hash==expected_source_code_sha256==plan['notebook_code_sha256'],'Reviewed notebook source changed')
    for path,digest in plan['input_sha256'].items():
        v027_require(v027_sha(root/path)==digest,'Pinned input changed: '+path)
    out=root/'outputs/neighbors_feasibility_v027'
    if out.exists():raise FileExistsError('Preserve existing diagnostic; review explicit recovery rather than overwrite')
    out.mkdir(parents=True);started=time.monotonic()
    preflight=v027_synthetic_preflight();v027_json(out/'synthetic_preflight.json',preflight)
    v027_json(out/'design.json',dict(protocol_sha256=expected_protocol_sha256,source_code_sha256=source_hash,recipe=V027_RECIPE,started_utc=datetime.now(timezone.utc).isoformat(),input_sha256=plan['input_sha256'],gradient_model_fits=0,neural_checkpoint_inferences=0,original_Aval_scored=False,training_pool_statistics_fitted='planned; not fitted at design creation'))
    with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz',allow_pickle=False) as split:
        fit_ids=split['A_fit'].copy();Aval_ids=split['A_validation'].copy()
    v027_require(len(fit_ids)==61114 and len(Aval_ids)==6791 and not np.intersect1d(fit_ids,Aval_ids).size,'Original A split changed')
    bids,x,y,row_ids,rc=v027_load_Afit_only(root/'data/train.csv',fit_ids)
    query,pool=v027_query_pool_indices(rc,V027_RECIPE['query_seed'],V027_RECIPE['queries_per_RC'])
    v027_require(not np.intersect1d(bids[query],Aval_ids).size and not np.intersect1d(bids[pool],Aval_ids).size,'Original Aval contamination')
    v027_json(out/'sample_before_scoring.json',dict(query_breath_ids=bids[query].tolist(),query_seed=V027_RECIPE['query_seed'],queries_per_RC=V027_RECIPE['queries_per_RC'],query_breaths=576,pool_breaths=60538,pool_breath_ids_array_sha256=v027_ah(bids[pool]),query_row_ids_array_sha256=v027_ah(row_ids[query]),original_Aval_ids_used_only_for_exclusion=True,query_labels_used_for_pool_statistics=False))
    # Limit BLAS threads to make bounded CPU distance evaluation reproducible.
    with threadpool_limits(limits=1):
        predicted=v027_predict_from_pool(x[query],x[pool],rc[query],rc[pool],y[pool])
    nn,distance,scaler,prediction,disagreement,spread,baseline,fallback=(predicted[k] for k in ['nn','distance','scaler','prediction','disagreement','spread','baseline','fallback'])
    truth=y[query].astype(np.float64);phase=x[query,:,2].astype(np.int8);mask=phase==0
    v027_require(mask.sum(axis=1).min()>0,'Query breath without scored inspiration')
    v027_require(prediction.shape==baseline.shape==truth.shape==(576,80),'Prediction coverage changed')
    e=np.abs(prediction-truth);base_e=np.abs(baseline-truth)
    counts=mask.sum(axis=1);error_sums=(e*mask).sum(axis=1);base_sums=(base_e*mask).sum(axis=1)
    breath_mae=error_sums/counts;mean_distance=distance.mean(axis=1)
    mean_disagreement=(disagreement*mask).sum(axis=1)/counts
    neighbor_phase=x[pool,:,2][nn]
    phase_mismatch=(neighbor_phase!=phase[:,None,:]).mean(axis=(1,2))
    inspiratory_phase_mismatch=((neighbor_phase!=phase[:,None,:])*mask[:,None,:]).sum(axis=(1,2))/(5*counts)
    # Only this section uses query targets. No retrieval/scaler fitting follows.
    rng=np.random.default_rng(V027_RECIPE['bootstrap_seed']);delta_sums=error_sums-base_sums;draws=[]
    for _ in range(V027_RECIPE['bootstrap_draws']):
        take=rng.integers(0,len(query),size=len(query));draws.append(float(delta_sums[take].sum()/counts[take].sum()))
    ci=np.quantile(draws,[.025,.975]);rc_table=[]
    rc_table.extend(v027_error_table(e,phase,rc[query],'top5_neighbor_median'))
    rc_table.extend(v027_error_table(base_e,phase,rc[query],'RC_position_inspiratory_median'))
    correlations=dict(mean_control_distance_vs_breath_MAE=v027_spearman(mean_distance,breath_mae),nearest_control_distance_vs_breath_MAE=v027_spearman(distance[:,0],breath_mae),neighbor_MAD_vs_point_absolute_error=v027_spearman(disagreement[mask],e[mask]),mean_neighbor_MAD_vs_breath_MAE=v027_spearman(mean_disagreement,breath_mae),phase_mismatch_vs_breath_MAE=v027_spearman(phase_mismatch,breath_mae))
    by_RC_correlations=[]
    for setting in sorted(set(map(tuple,rc[query]))):
        select=np.all(rc[query]==setting,axis=1)
        by_RC_correlations.append(dict(R=int(setting[0]),C=int(setting[1]),queries=int(select.sum()),mean_neighbor_phase_mismatch_fraction=float(phase_mismatch[select].mean()),mean_inspiratory_neighbor_phase_mismatch_fraction=float(inspiratory_phase_mismatch[select].mean()),distance_vs_breath_MAE=v027_spearman(mean_distance[select],breath_mae[select]),disagreement_vs_breath_MAE=v027_spearman(mean_disagreement[select],breath_mae[select])))
    per_query=pd.DataFrame(dict(breath_id=bids[query],R=rc[query,0],C=rc[query,1],inspiratory_points=counts,neighbor_MAE=breath_mae,baseline_MAE=base_sums/counts,nearest_control_distance=distance[:,0],mean_control_distance=mean_distance,fifth_control_distance=distance[:,-1],mean_neighbor_pressure_MAD=mean_disagreement,mean_neighbor_pressure_IQR=(spread*mask).sum(axis=1)/counts,neighbor_phase_mismatch_fraction=phase_mismatch,inspiratory_neighbor_phase_mismatch_fraction=inspiratory_phase_mismatch))
    for j in range(5):per_query['neighbor_'+str(j+1)+'_breath_id']=bids[pool][nn[:,j]]
    per_query.to_csv(out/'per_query.csv',index=False)
    pd.DataFrame(rc_table).to_csv(out/'by_RC.csv',index=False)
    pd.DataFrame(by_RC_correlations).to_csv(out/'by_RC_correlations.csv',index=False)
    arrays={'query_breath_ids.npy':bids[query],'pool_breath_ids.npy':bids[pool],'query_row_ids.npy':row_ids[query],'neighbor_pool_indices.npy':nn,'neighbor_distances.npy':distance,'neighbor_predictions.npy':prediction,'baseline_predictions.npy':baseline,'neighbor_pressure_MAD.npy':disagreement,'neighbor_pressure_IQR.npy':spread,'query_inspiratory_mask.npy':mask}
    for filename,value in arrays.items():v027_array(out/filename,value)
    summary=dict(status='complete_training_only_diagnostic',completed_utc=datetime.now(timezone.utc).isoformat(),runtime_seconds=time.monotonic()-started,protocol_sha256=expected_protocol_sha256,source_code_sha256=source_hash,recipe=V027_RECIPE,query_breaths=576,pool_breaths=60538,scored_points=int(mask.sum()),query_coverage=1.,native_inspiratory_MAE=float(e[mask].mean()),baseline_native_inspiratory_MAE=float(base_e[mask].mean()),candidate_minus_baseline=float(delta_sums.sum()/counts.sum()),paired_breath_bootstrap_95=[float(ci[0]),float(ci[1])],median_absolute_error=float(np.median(e[mask])),baseline_median_absolute_error=float(np.median(base_e[mask])),primary_comparison_metric='native inspiratory mean MAE; all other metrics descriptive only',pressure_disagreement_is_not_calibrated_uncertainty=True,scaler=scaler,baseline_empty_cell_fallback_steps=fallback,correlations=correlations,distance_quartile_diagnostics=v027_bins(mean_distance,error_sums,counts,4),disagreement_quartile_diagnostics=v027_bins(mean_disagreement,error_sums,counts,4),phase_mismatch_mean=float(phase_mismatch.mean()),gradient_model_fits=0,neural_checkpoint_inferences=0,no_original_Aval_outcome=True,no_test_data=True,no_F_split_or_fit=True,no_submission=True,automatic_GPU_promotion=False,training_pool_statistics_fitted=True,nonparametric_estimators_evaluated=2,interpretation='This training-only inner holdout compares one fixed KNN estimator with a pool-only RC/position baseline. The RC-position reference is a deliberately simple and weak baseline; beating it does not establish improvement over the LSTM, original A, cold F, the official score, or a pressure-MAE target. A positive result supports only a separately frozen, cross-fitted residual-context experiment; no automatic GPU action.',runtime=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,BLAS=threadpool_info()),array_sha256={k:v027_ah(v) for k,v in arrays.items()})
    # Accepted submission is an immutable preservation check, not a tuning signal.
    for path,digest in plan['preserved_files_sha256'].items():v027_require(v027_sha(root/path)==digest,'Preserved accepted artifact changed')
    summary['artifact_sha256']={p.name:v027_sha(p) for p in sorted(out.iterdir()) if p.is_file()}
    v027_json(out/'summary.json',summary)
    print(json.dumps({k:summary[k] for k in ['status','native_inspiratory_MAE','baseline_native_inspiratory_MAE','candidate_minus_baseline','paired_breath_bootstrap_95','runtime_seconds','automatic_GPU_promotion']},indent=2))
    return summary


## Review before execution

The companion `neighbors_protocol_v027_DRAFT.json` is deliberately rejected by the runner because its status is `draft`. Root must review the design and, if accepted, create a separately frozen protocol with the exact source/input hashes. Then call `run_neighbors_feasibility_v027` explicitly with the reviewed frozen protocol SHA and notebook **code** SHA. No environment variable or notebook cell starts it automatically.

The output directory must be new. An interrupted diagnostic is preserved for explicit recovery review rather than overwritten. This source and draft have undergone syntax compilation only; no input arrays, distances, scores, correlations or outcomes have been calculated.


In [5]:
import os
if os.environ.get('PA3_RUN_V027','0')=='1':
    V027_PINS=json.loads((Path.cwd()/'research/freeze_v027.json').read_text())
    V027_RESULT=run_neighbors_feasibility_v027(Path.cwd(),expected_protocol_sha256=V027_PINS['protocol_sha256'],expected_source_code_sha256=V027_PINS['source_code_sha256'])


{
  "status": "complete_training_only_diagnostic",
  "native_inspiratory_MAE": 0.7197577777812822,
  "baseline_native_inspiratory_MAE": 6.061132637798443,
  "candidate_minus_baseline": -5.341374860017161,
  "paired_breath_bootstrap_95": [
    -5.6766654282376505,
    -5.034357111120914
  ],
  "runtime_seconds": 24.95839854099995,
  "automatic_GPU_promotion": false
}
